In [38]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Imports

In [32]:
import os
import sys

sys.path.append("..")
sys.path.append("./ALAE")

import random

import numpy as np
import torch
from tqdm import tqdm

import wandb
from src.costs.lse import MLPLSECost
from src.models.gmm_based import GMMEOT
from src.models.light_sbm import LightSBM
from src.plotting.parameters import (
    plot_A_parameters,
    plot_B_parameters,
    plot_Z_parameters,
)
from src.samplers.from_dataset import DatasetSampler
from src.utils.train import compute_loss, update_average

In [3]:
device = torch.device(f"cuda:{torch.cuda.current_device()}" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

In [4]:
torch.set_default_device(device)
# dtype = torch.float64
dtype = torch.float32
# torch.torch.set_default_dtype(dtype)

## 2. Config

In [5]:
from configs.gmm_based.cost import MLPLSECostConfig
from configs.gmm_based.optimizer import OptPairedConfig, OptUnpairedConfig
from configs.gmm_based.train import TrainConfig

In [120]:
# Data Type
X_DIM = 512
Y_DIM = 512
INPUT_DATA = "ADULT" # MAN, WOMAN, ADULT, CHILDREN
TARGET_DATA = "CHILDREN" # MAN, WOMAN, ADULT, CHILDREN

# Data
Q_X_UNPAIRED_SAMPLES = 48786 # 1024
R_Y_UNPAIRED_SAMPLES = 10762 # 1024
P_XY_PAIRED_SAMPLES = 5600 # 128

# Optimizer
LR_PAIRED = 3e-4
LR_UNPAIRED = 3e-4

# Sampler
PAIRED_BATCH_SIZE = 128
UNPAIRED_BATCH_SIZE = 128

# Train
MAX_STEPS = 10000
INIT_BY_SAMPLES = True

# Potential
N_POTENTIALS = 5

# Cost
M_POTENTIALS = 3
LOG_V_M_HIDDEN_CHANNELS = [M_POTENTIALS]
B_M_HIDDEN_CHANNELS = [M_POTENTIALS * Y_DIM]

In [121]:
cost_config = MLPLSECostConfig(
    x_dim=X_DIM,
    y_dim=Y_DIM,
    m_potentials=M_POTENTIALS,
    log_v_m_hidden_channels=LOG_V_M_HIDDEN_CHANNELS,
    b_m_hidden_channels=B_M_HIDDEN_CHANNELS,
)
EXP_META_INFO = (
    f"M_POTENTIALS_{M_POTENTIALS}_"
    + f"LOG_V_M_HIDDEN_CHANNELS_{LOG_V_M_HIDDEN_CHANNELS}_"
    + f"B_M_HIDDEN_CHANNELS_{B_M_HIDDEN_CHANNELS}_"
)

opt_unpaired_config = OptUnpairedConfig(lr=LR_UNPAIRED)
opt_paired_config = OptPairedConfig(lr=LR_PAIRED)

train_config = TrainConfig(
    steps_to=MAX_STEPS, paired_batch_size=PAIRED_BATCH_SIZE, unpaired_batch_size=UNPAIRED_BATCH_SIZE
)

In [122]:
torch.manual_seed(train_config.seed)
np.random.seed(train_config.seed)
random.seed(train_config.seed)

## 3. Create data and samplers

In [123]:
from src.utils.datasets import get_latents
from src.samplers.base import TensorSampler
from src.utils.paired import get_paired_sampler

In [124]:
X_train, X_test = get_latents("ADULT", dtype=dtype)
Y_train, Y_test = get_latents("CHILDREN", dtype=dtype)

In [125]:
X_sampler = TensorSampler(X_train.to(dtype), device=device)
Y_sampler = TensorSampler(Y_train.to(dtype), device=device)

In [126]:
from_dir = f"./datasets/FFHQ/pairs/{INPUT_DATA}->{TARGET_DATA}"
X_paired_train = torch.load(os.path.join(from_dir, f"X_train.pt"), map_location=device, weights_only=True).to(dtype)
Y_paired_train = torch.load(os.path.join(from_dir, f"Y_train.pt"), map_location=device, weights_only=True).to(dtype)

X_paired_test = torch.load(os.path.join(from_dir, f"X_test.pt"), map_location=device, weights_only=True).to(dtype)
Y_paired_test = torch.load(os.path.join(from_dir, f"Y_test.pt"), map_location=device, weights_only=True).to(dtype)

In [127]:
pd_train_sampler = get_paired_sampler(
    X_paired_train, Y_paired_train, train_config.paired_batch_size, P_XY_PAIRED_SAMPLES, device
)

In [128]:
if Q_X_UNPAIRED_SAMPLES > 0:
    source_data = X_sampler.sample(Q_X_UNPAIRED_SAMPLES)
    usd_sampler = DatasetSampler(source_data, device=device) # usd - unpaired source data
else:
    usd_sampler = DatasetSampler(X_paired_train, device=device)

if R_Y_UNPAIRED_SAMPLES > 0:
    target_data = Y_sampler.sample(R_Y_UNPAIRED_SAMPLES)
    utd_sampler = DatasetSampler(target_data, device=device) # utd - unpaired target data
else:
    utd_sampler = DatasetSampler(Y_paired_train, device=device)

## 4. Model initialization

In [129]:
cost = MLPLSECost(**cost_config.model_dump())

In [130]:
model = GMMEOT(
    y_dim=Y_DIM,
    n_potentials=N_POTENTIALS,
    cost=cost,
).to(dtype)

if INIT_BY_SAMPLES:
    model.init_a_by_samples(Y_sampler.sample(N_POTENTIALS))

In [131]:
# For EMA update
if train_config.ema_update:
    model_copy = GMMEOT(
    y_dim=Y_DIM,
    n_potentials=N_POTENTIALS,
    cost=cost,
).to(dtype)

## 5. Optimizers initialization

In [132]:
unpaired_params_to_update = [model._log_w_n, model._a_n, model._log_A_n]

D_opt_unpaired = torch.optim.Adam(unpaired_params_to_update, **opt_unpaired_config.model_dump())

In [133]:
D_opt_paired = torch.optim.Adam(model.cost.parameters(), **opt_paired_config.model_dump())

In [134]:
# TODO: refactor this config
EXP_NAME = (
    "GMMEOT_ALAE_"
    + f"FROM_{INPUT_DATA}_"
    + f"TO_{TARGET_DATA}_"
    + f"P_XY_PAIRED_{P_XY_PAIRED_SAMPLES}_"
    + f"Q_X_UNPAIRED_{Q_X_UNPAIRED_SAMPLES}_"
    + f"R_Y_UNPAIRED_{R_Y_UNPAIRED_SAMPLES}_"
    + f"LR_PAIRED_{opt_paired_config.lr}_"
    + f"LR_UNPAIRED_{opt_unpaired_config.lr}_"
    + EXP_META_INFO
)
OUTPUT_PATH = "../checkpoints/{}".format(EXP_NAME)

config = dict(
    D_LR_PAIRED=opt_paired_config.lr,
    D_LR_UNPAIRED=opt_unpaired_config.lr,
    BATCH_SIZE=train_config.unpaired_batch_size,
    P_XY_PAIRED_SAMPLES=P_XY_PAIRED_SAMPLES,
    Q_X_UNPAIRED_SAMPLES=Q_X_UNPAIRED_SAMPLES,
    R_Y_UNPAIRED_SAMPLES=R_Y_UNPAIRED_SAMPLES,
)

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH, exist_ok=True)

In [135]:
if train_config.steps_from > 0:
    D_opt_unpaired.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{train_config.steps_from}.pt")))
    D_opt_paired.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_paired_{train_config.steps_from}.pt")))

## 6. Model training

In [136]:
starting_points = X_test[:3]
num_ending_points = 64

In [137]:
num_starting_points_paired = 5
indices = random.choices(range(P_XY_PAIRED_SAMPLES), k=num_starting_points_paired)
starting_points_paired = X_paired_train[indices]
ending_points_paired = Y_paired_train[indices]

In [138]:
wandb.init(name=EXP_NAME, config=config)

for step in tqdm(range(train_config.steps_from, train_config.steps_to)):
    # training loop
    D_opt_unpaired.zero_grad()

    X = usd_sampler.sample(train_config.unpaired_batch_size)
    Y = utd_sampler.sample(train_config.unpaired_batch_size)

    output_unpaired = model.compute_unpaired_loss(X, Y)
    D_loss_unpaired = output_unpaired["loss"]

    wandb.log({f"Unpaired loss": D_loss_unpaired.item()}, step=step)

    D_opt_paired.zero_grad()
    X_paired, Y_paired = pd_train_sampler.sample(train_config.paired_batch_size)
    
    output_paired = model.compute_paired_loss(X_paired, Y_paired)
    D_loss_paired = output_paired["loss"]

    wandb.log({f"Paired loss": D_loss_paired.item()}, step=step)

    D_loss = D_loss_unpaired + D_loss_paired
    D_loss.backward()
    D_opt_paired.step()
    D_opt_unpaired.step()

    if train_config.ema_update:
        update_average(model_copy, model, 0.99)
        model = model_copy
    else:
        model = model

    wandb.log({f"Loss": D_loss}, step=step)
    wandb.log(
        {f"Train paired loss": compute_loss(model, X_paired_train, Y_paired_train, X_paired_train, Y_paired_train)},
        step=step,
    )
    wandb.log(
        {f"Test paired loss": compute_loss(model, X_paired_test, Y_paired_test, X_paired_test, Y_paired_test)},
        step=step,
    )
    # wandb.log(
    #     {f"Test unpaired loss": compute_loss(model, X_unpaired_test, Y_unpaired_test, X_paired_test, Y_paired_test)},
    #     step=step,
    # )

    wandb.log({r"$-f^c(x)$": -output_unpaired["f_c"].mean().item()}, step=step)
    wandb.log({r"$-f(y)$": -output_unpaired["f"].mean().item()}, step=step)
    wandb.log({f"lam_min(A_n)": torch.min(output_unpaired["A_n"])}, step=step)
    wandb.log({f"lam_max(A_n)": torch.max(output_unpaired["A_n"])}, step=step)

    if step % train_config.plot_every == 0:
        A_dict = plot_A_parameters(model, log=True)
        B_dict = plot_B_parameters(model.cost, starting_points, log=True)
        if num_starting_points_paired > 0:
            Z_dict = plot_Z_parameters(model, starting_points, starting_points_paired, ending_points_paired, log=True)
        else:
            Z_dict = plot_Z_parameters(model, starting_points, log=True)
        wandb.log(A_dict | B_dict | Z_dict)
        wandb.log(A_dict)

        torch.save(model.state_dict(), os.path.join(OUTPUT_PATH, f"D_{step}.pt"))

torch.save(model.state_dict(), os.path.join(OUTPUT_PATH, f"D_{MAX_STEPS}.pt"))
torch.save(D_opt_paired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_paired_{MAX_STEPS}.pt"))
torch.save(D_opt_unpaired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{MAX_STEPS}.pt"))

wandb.finish()

$-f(y)$,█▇▅▅▅▄▄▃▃▃▂▁▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▃▃▂▃▂▃▃▃
$-f^c(x)$,▁▂▃▅▅▆▇▆▇█▆▇▇▇██▇▇▇█▆▇▇██▇▇███▇▇█▇█▇█▇▇█
Loss,█▆▄▄▄▄▃▂▃▄▂▂▂▂▃▃▃▂▂▂▂▂▂▂▂▂▂▃▁▂▁▁▂▁▁▂▁▂▂▂
Paired loss,█▆▅▄▄▃▂▃▃▃▃▃▂▂▂▂▃▂▂▂▃▂▂▂▂▂▂▂▁▂▂▁▁▁▁▂▁▁▂▂
Test paired loss,█▆▄▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Train paired loss,█▆▄▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Unpaired loss,▂▁▁▃▅▅▆▄▅▇▃▂▄▅▇▆▅▆▆▆▂▅▅▆▆▅▅▇▇▆▅▆█▆▆▅▆▇▇█
lam_max(A_n),▁▁▁▂▂▂▃▃▃▃▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇████████
lam_min(A_n),██▇▆▆▅▅▅▄▄▄▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
$-f(y)$,1053.09265
$-f^c(x)$,564.92676


  4%|██████▋                                                                                                                                                              | 409/10000 [00:14<04:20, 36.78it/s]wandb: WARNING Step only supports monotonically increasing values, use define_metric to set a custom x axis. For details see: https://wandb.me/define-metric
wandb: WARNING (User provided step: 1 is less than current step: 2. Dropping entry: {'Unpaired loss': 1444.5079345703125, '_timestamp': 1732714685.1832309}).
wandb: WARNING (User provided step: 1 is less than current step: 2. Dropping entry: {'Paired loss': -15.72916030883789, '_timestamp': 1732714685.18681}).
wandb: WARNING (User provided step: 1 is less than current step: 2. Dropping entry: {'Loss': 1428.77880859375, '_timestamp': 1732714685.1911955}).
wandb: WARNING (User provided step: 1 is less than current step: 2. Dropping entry: {'Train paired loss': 1480.2708740234375, '_timestamp': 1732714685.2008448}).
wandb: WARNING (

KeyboardInterrupt: 

# 7. Light-SBM training

In [ ]:
import torch.nn.functional as F

In [33]:
eps = 0.1
lr = 1e-3

n_potentials = 10
is_diag = True
S_init = 0.1

max_iter = 20000

In [34]:
light_sbm = LightSBM(dim=X_DIM, n_potentials=n_potentials, epsilon=eps, S_diagonal_init=S_init, is_diagonal=is_diag)

light_sbm.to(device)
light_sbm_opt = torch.optim.Adam(light_sbm.parameters(), lr=lr)

In [39]:
def train(model, max_iter, eps, opt, val_freq=1000, batch_size=512, safe_t=1e-2, device=device):
    
    pbar = tqdm(range(1, max_iter + 1))
    
    for i in pbar:
        
        x_0_samples = X_sampler.sample(batch_size).to(device)      
        x_1_samples = Y_sampler.sample(batch_size).to(device)
        
        t = torch.rand([batch_size, 1], device=device) * (1 - safe_t)
        
        x_t = x_1_samples * t + x_0_samples * (1 - t) + torch.sqrt(eps * t * (1 - t)) * torch.randn_like(x_0_samples)
                
        predicted_drift = model.get_drift(x_t, t.squeeze())
        
        loss_plan = (model.get_log_C(x_0_samples) - model.get_log_potential(x_1_samples)).mean()
        
        target_drift = (x_1_samples - x_t) / (1 - t)
        
        loss = F.mse_loss(target_drift, predicted_drift)
        
        opt.zero_grad()
        
        loss.backward()
        
        opt.step()
        
        pbar.set_description(f'Loss : {loss.item()} Plan Loss: {loss_plan.item()}')
        
        if wandb.run:
            wandb.log({'loss_bm': loss, 'loss_plan': loss_plan})
        
        if i % val_freq == 0:
            pass

In [40]:
train(light_sbm, max_iter, eps, light_sbm_opt, val_freq=1000, batch_size=512, safe_t=1e-2, device=device)

  0%|                                                                                                                                                                               | 0/20000 [00:00<?, ?it/s]/trinity/home/m.persiyanov/miniconda3/envs/text/lib/python3.11/site-packages/torch/utils/_device.py:106: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return func(*args, **kwargs)
Loss : 1.35252046585083 Plan Loss: 4168.6982421875: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20000/20000 [03:02<00:00, 109.51it/s]


# 8. Metrics

In [87]:
from alae_ffhq_inference import decode, load_model
from torchmetrics.image import StructuralSimilarityIndexMeasure
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity

In [139]:
alae_model = load_model("./ALAE/configs/ffhq.yaml", training_artifacts_dir="./ALAE/training_artifacts/ffhq/").to(
    device
).to(dtype)

model.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_{3000}.pt"), map_location=device, weights_only=True))

/beegfs/home/m.persiyanov/Light-GCOT/./ALAE/checkpointer.py:92: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(f, map_location=torch.device("cpu"))


<All keys matched successfully>

In [89]:
def normalize_tensor(tensor: torch.Tensor) -> torch.Tensor:
    normalized = tensor / 2 + 0.5
    return normalized.clamp_(0, 1)

def to_uint8(normalized_tensor: torch.Tensor) -> torch.Tensor:
    return normalized_tensor.mul(255).add_(0.5).clamp_(0, 255).to(torch.uint8)

In [90]:
def eval_model(model: torch.nn.Module) -> tuple[float, float, float]:
    loss_fid = FrechetInceptionDistance().to(device)
    loss_ssim = StructuralSimilarityIndexMeasure(data_range=(-1.0, 1.0)).to(device)
    loss_lpip = LearnedPerceptualImagePatchSimilarity(net_type='alex').to(device)
    
    with torch.no_grad():
        sampling_batch_size = 128
        num_samples = len(X_paired_test)
        
        num_sampling_iterations = (
            num_samples // sampling_batch_size
            if num_samples % sampling_batch_size == 0
            else (num_samples // sampling_batch_size) + 1
        )
        for i in tqdm(range(num_sampling_iterations)):
            sub_batch_x = X_paired_test[sampling_batch_size * i : sampling_batch_size * (i + 1)]
            sub_batch_y = Y_paired_test[sampling_batch_size * i : sampling_batch_size * (i + 1)]

            y_pred = model(sub_batch_x)
            normalized_pred_images = normalize_tensor(decode(alae_model, y_pred))
            normalized_true_images = normalize_tensor(decode(alae_model, sub_batch_y))

            loss_fid.update(to_uint8(normalized_pred_images), real=True)
            loss_fid.update(to_uint8(normalized_true_images), real=False)

            loss_ssim.update(normalized_pred_images, normalized_true_images)
            loss_lpip.update(normalized_pred_images, normalized_true_images)

            # Explicitly free sub-batches to release GPU memory
            del sub_batch_x, sub_batch_y, y_pred, normalized_pred_images, normalized_true_images
            torch.cuda.empty_cache()

    return loss_fid.compute(), loss_ssim.compute(), loss_lpip.compute()

In [140]:
loss_fid, loss_ssim, loss_lpip = eval_model(model)
print(f"FID: {loss_fid}")
print(f"SSIM: {loss_ssim}")
print(f"LPIPS: {loss_lpip}")

/trinity/home/m.persiyanov/miniconda3/envs/text/lib/python3.11/site-packages/torchmetrics/functional/image/lpips.py:323: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.l

FID: 9.038664817810059
SSIM: 0.6832053065299988
LPIPS: 0.3153505325317383


In [92]:
loss_fid_light_sbm, loss_ssim_light_sbm, loss_lpip_light_sbm = eval_model(light_sbm)
print(f"FID: {loss_fid_light_sbm}")
print(f"SSIM: {loss_ssim_light_sbm}")
print(f"LPIPS: {loss_lpip_light_sbm}")

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 24/24 [01:24<00:00,  3.54s/it]


FID: 7.638559818267822
SSIM: 0.7087258100509644
LPIPS: 0.26895761489868164


# 8. Plotting

In [69]:
from matplotlib import pyplot as plt

In [71]:
decoded_img_np = decoded_img.cpu().permute(0, 2, 3, 1).numpy()
true_img_np = true_img.cpu().permute(0, 2, 3, 1).numpy()

In [75]:
fig, axes = plt.subplots(10, 2, figsize=(1, 5), dpi=200)

for i, ind in enumerate(range(10)):
    ax = axes[i]
    ax[0].imshow(true_img_np[ind])
    for k in range(1):
        ax[k+1].imshow(decoded_img_np[ind])
        
        ax[k+1].get_xaxis().set_visible(False)
        ax[k+1].set_yticks([])
        
    ax[0].get_xaxis().set_visible(False)
    ax[0].set_yticks([])

fig.tight_layout(pad=0.05)
fig.savefig('gmmeot_transfer.png')